# Assignment 3
## Econ 8310 - Business Forecasting

For homework assignment 3, you will work with [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist), a more fancier data set.

- You must create a custom data loader as described in the first week of neural network lectures [2 points]
    - You will NOT receive credit for this if you use the pytorch prebuilt loader for Fashion MNIST!
- You must create a working and trained neural network using only pytorch [2 points]
- You must store your weights and create an import script so that I can evaluate your model without training it [2 points]

Highest accuracy score gets some extra credit!

Submit your forked repository URL on Canvas! :) I'll be manually grading this assignment.

Some checks you can make on your own:
- Did you manually process the data or use a prebuilt loader (see above)?
- Does your script train a neural network on the assigned data?
- Did your script save your model?
- Do you have separate code to import your model for use after training?

In [20]:

#!pip3 install torch torchvision torchaudio
#!pip3 install plotly.express
#!pip install opencv-python
#!pip install "numpy<2"
!pip install tqdm



In [21]:
import torch
print(torch.cuda.is_available())

False


In [9]:
# For reading data
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import os
import cv2
import xml.etree.ElementTree as ET
import numpy as np

# For visualizing
import plotly.express as px

# For model building
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Defining our data class

class BaseballProj(Dataset):
    def __init__(self, root, transform = None):
        # set folder location
        self.video_folder = os.path.join(root, "Raw Videos")
        self.anno_folder = os.path.join(root, "Annotations")
        self.transform = transform

        anno = sorted(os.listdir(self.anno_folder))

        #create paths for each existing annotation and corresponding video
        self.anno_paths = []
        self.video_paths = []
        for file in anno:
            file_name = os.path.splitext(file)[0]

            anno_path = os.path.join(self.anno_folder, file)
            video_path = os.path.join(self.video_folder, f"{file_name}.mov")

            self.anno_paths.append(anno_path)
            self.video_paths.append(video_path)

    def __len__(self):
        return len(self.anno_paths)

    # retrieve a single record based on index position `idx`
    def __getitem__(self, idx):
        vid_path = self.video_paths[idx]
        vid = cv2.VideoCapture(vid_path)

        std_size = 3840
        scale_size = 960
        std_frames = 48

        #load video data
        frames = []
        while vid.isOpened():
            ret, frame = vid.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w, channels = frame.shape
            
            square_frame = np.zeros((std_size, std_size, channels), dtype=np.uint8)
            
            # Paste the original frame into the top-left corner
            square_frame[0:h, 0:w] = frame
            
            # Resize down to 1024x1024 to save GPU memory
            resized_frame = cv2.resize(square_frame, (scale_size, scale_size))
            
            frames.append(resized_frame)
            
        vid.release()


        #load bonding boxes
        scale_ratio = scale_size / std_size

        xml_path = self.anno_paths[idx]
        tree = ET.parse(xml_path)
        root = tree.getroot()

        ball_dict = {}
        
        for track in root.findall('track'):
            if track.get('label') == 'baseball':
                for box in track.findall('box'):
                    
                    is_moving = False
                    for attr in box.findall('attribute'):
                        if attr.get('name') == 'moving' and attr.text == 'true':
                            is_moving = True
                            break
                    
                    is_visible = box.get('outside') == '0'
                    
                    if is_moving and is_visible:
                        frame_num = int(box.get('frame'))
                        xtl = float(box.get('xtl')) * scale_ratio
                        ytl = float(box.get('ytl')) * scale_ratio
                        xbr = float(box.get('xbr')) * scale_ratio
                        ybr = float(box.get('ybr')) * scale_ratio

                        ball_dict[frame_num] = [xtl, ytl, xbr, ybr]


        #adding filler frames
        target_boxes = []
        target_labels = []
        
        for i in range(len(frames)):
            if i in ball_dict:
                target_boxes.append(ball_dict[i])
                target_labels.append(1)
            else:
                target_boxes.append([0.0, 0.0, 0.0, 0.0])
                target_labels.append(0)

        if len(frames) > std_frames:
            frames = frames[:std_frames]
            target_boxes = target_boxes[:std_frames]
            target_labels = target_labels[:std_frames]

        while len(frames) < std_frames:
            # Note: Make sure the zeros array matches your 960 target size!
            black_frame = np.zeros((960, 960, 3), dtype=np.uint8)
            frames.append(black_frame)
            target_boxes.append([0.0, 0.0, 0.0, 0.0])
            target_labels.append(0)
        

        frames_tensor = torch.stack([torch.from_numpy(f) for f in frames])
        
        boxes_tensor = torch.tensor(target_boxes, dtype=torch.float32)
        
        labels_tensor = torch.tensor(target_labels, dtype=torch.int64)

        if self.transform:
            frames_tensor, boxes_tensor, labels_tensor = self.transform(
                frames_tensor, boxes_tensor, labels_tensor
            )
            
        return frames_tensor, boxes_tensor, labels_tensor

        

Notes for final project
trim the video to only keep 1 frame before and after moving ball to save memory

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create separate dataloaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [17]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_baseball_tracker_model():
    # 1. Load the pre-trained Faster R-CNN model
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    
    # 2. We need 2 classes (Background + Moving Baseball)
    num_classes = 2 
    
    # 3. Get the number of input features for the final classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # 4. Replace the pre-trained "head" with a new, untrained one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

In [ ]:
import torch
from tqdm import tqdm

# 1. Initialize Model and push to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_baseball_tracker_model().to(device)

# 2. Set up Optimizer (AdamW is the modern standard)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=0.0001, weight_decay=0.0005)

# Number of times to loop over your entire dataset
num_epochs = 10 

print("Starting Training...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    epoch_loss = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_frames, batch_boxes, batch_labels in progress_bar:
        
        # ==========================================
        # FORMATTING FOR FASTER R-CNN
        # ==========================================
        # Get current dimensions: [Batch, Channels, Frames, Height, Width]
        B, C, F, H, W = batch_frames.shape
        
        # Flatten Batch and Frames into a single "Total Images" dimension
        # New shape: [B * F, C, H, W]
        images_tensor = batch_frames.permute(0, 2, 1, 3, 4).reshape(-1, C, H, W)
        
        # Flatten the boxes and labels arrays
        # New shapes: [B * F, 4] and [B * F]
        flat_boxes = batch_boxes.reshape(-1, 4)
        flat_labels = batch_labels.reshape(-1)
        
        # Faster R-CNN requires a list of images (normalized 0.0 to 1.0)
        # We divide by 255.0 to convert our 0-255 pixels into proper decimals
        images = list(img.to(device, dtype=torch.float32) / 255.0 for img in images_tensor)
        
        # Faster R-CNN requires a list of dictionaries for the targets
        targets = []
        for i in range(len(images)):
            target = {}
            # Need to reshape from [4] to [1, 4] to represent one box
            target["boxes"] = flat_boxes[i].unsqueeze(0).to(device)
            target["labels"] = flat_labels[i].unsqueeze(0).to(device)
            targets.append(target)

        # ==========================================
        # THE CORE ML MATH
        # ==========================================
        optimizer.zero_grad()
        
        # Faster R-CNN is unique: during training, it automatically calculates 
        # and returns the losses directly!
        loss_dict = model(images, targets)
        
        # Sum up the bounding box loss and classification loss
        losses = sum(loss for loss in loss_dict.values())
        
        # Calculate gradients and update weights
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        
        # Print progress every batch
        print(f"Epoch {epoch+1} | Batch {batch_idx+1} | Loss: {losses.item():.4f}")
        
    print(f"--- Epoch {epoch+1} Completed. Average Loss: {epoch_loss / len(dataloader):.4f} ---")

print("Training Complete!")

# Save your trained model!
torch.save(model.state_dict(), "baseball_tracker_model.pth")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to C:\Users\Alan/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth
100.0%


Starting Training...


IndexError: index 480 is out of bounds for dimension 0 with size 480